In [15]:
from halox import emus, hmf, lss, bias
import jax.numpy as jnp
import jax_cosmo as jc
import jax
import time

# Quantifying acceleration of sigmaM computation

The aim of this notebook is to display the change in computation speed for calculations of sigmaM, the halo mass function, and halo bias using a neural network to compute sigmaM.

In [6]:
cosmo_fid = jc.Planck15()

cosmo_high_s8 = jc.Cosmology(
    Omega_b=cosmo_fid.Omega_b,
    Omega_c=cosmo_fid.Omega_c,
    h=cosmo_fid.h,
    sigma8=0.9,
    n_s=cosmo_fid.n_s,
    Omega_k=0,
    w0=-1,
    wa=0,
)

cosmo_low_s8 = jc.Cosmology(
    Omega_b=cosmo_fid.Omega_b,
    Omega_c=cosmo_fid.Omega_c,
    h=cosmo_fid.h,
    sigma8=0.7,
    n_s=cosmo_fid.n_s,
    Omega_k=0,
    w0=-1,
    wa=0,
)
c_list = [cosmo_fid, cosmo_high_s8, cosmo_low_s8]
cosmos = emus.sigmaM.stack_cosmologies(c_list)
print(jnp.shape(cosmos.Omega_b))

(3,)


### $\sigma$(M)

In [8]:
emu = emus.sigmaM.SigmaMEmulator()

masses = jnp.logspace(11, 15, 256)
zs = jnp.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0])

halox_sm_vmap = jax.jit(
    jax.vmap(
        jax.vmap(
            lambda M, z, cosmo: lss.sigma_M(M, z, cosmo),
            in_axes=(None, 0, None),
        ),
        in_axes=(None, None, 0),
    )
)
emu_sm_vmap = jax.jit(
    jax.vmap(
        jax.vmap(
            lambda M, z, cosmo: emu(M, z, cosmo), in_axes=(None, 0, None)
        ),
        in_axes=(None, None, 0),
    )
)


def benchmark(fn, Ms, zs, cosmos, n=1000):
    fn = jax.jit(fn)
    # warmup
    fn(Ms, zs, cosmos).block_until_ready()

    start = time.perf_counter()

    def loop(i, _):
        return fn(Ms, zs, cosmos)

    start = time.perf_counter()

    y = jax.lax.fori_loop(0, n, loop, fn(Ms, zs, cosmos))
    jax.block_until_ready(y)

    end = time.perf_counter()

    return (end - start) / n


t_nn = benchmark(emu_sm_vmap, masses, zs, cosmos)
t_true = benchmark(halox_sm_vmap, masses, zs, cosmos)

print("NN time per call:", t_nn)
print("Analytic time per call:", t_true)
print("Speedup:", t_true / t_nn)

NN time per call: 0.00015285620599752293
Analytic time per call: 0.020601561213989043
Speedup: 134.77739473870557


In [10]:
%timeit emu_sm_vmap(masses, zs, cosmos).block_until_ready()
%timeit halox_sm_vmap(masses, zs, cosmos).block_until_ready()

6.24 ms ± 328 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
176 ms ± 11.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Halo Mass Function (Tinker et al. 2008)

In [11]:
halox_mf_vmap = jax.jit(
    jax.vmap(
        jax.vmap(
            lambda M, z, cosmo: hmf.tinker08_mass_function(M, z, cosmo),
            in_axes=(None, 0, None),
        ),
        in_axes=(None, None, 0),
    )
)
emu_mf_vmap = jax.jit(
    jax.vmap(
        jax.vmap(
            lambda M, z, cosmo: hmf.tinker08_mass_function(
                M, z, cosmo, emulate=True
            ),
            in_axes=(None, 0, None),
        ),
        in_axes=(None, None, 0),
    )
)

In [12]:
%timeit emu_mf_vmap(masses, zs, cosmos).block_until_ready()
%timeit halox_mf_vmap(masses, zs, cosmos).block_until_ready()

19.2 ms ± 529 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
302 ms ± 47.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Halo Bias (Tinker et al. 2010)

In [16]:
halox_bi_vmap = jax.jit(
    jax.vmap(
        jax.vmap(
            lambda M, z, cosmo: bias.tinker10_bias(M, z, cosmo),
            in_axes=(None, 0, None),
        ),
        in_axes=(None, None, 0),
    )
)
emu_bi_vmap = jax.jit(
    jax.vmap(
        jax.vmap(
            lambda M, z, cosmo: bias.tinker10_bias(M, z, cosmo, emulate=True),
            in_axes=(None, 0, None),
        ),
        in_axes=(None, None, 0),
    )
)

In [17]:
%timeit emu_bi_vmap(masses, zs, cosmos).block_until_ready()
%timeit halox_bi_vmap(masses, zs, cosmos).block_until_ready()

7.89 ms ± 154 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
209 ms ± 32.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
